# Notebook 01 — Exploratory Data Analysis

**ECG Arrhythmia Classification using Machine Learning**

This notebook performs comprehensive EDA on the MIT-BIH Arrhythmia Database.

---
**Data Integrity Rule**: No preprocessing is applied in this notebook. We explore the raw segmented data to understand class distributions, beat morphology, and RR interval patterns.

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data_loader import AAMI_CLASSES, load_dataset_metadata
from src.utils import get_records, load_config
from src.visualization import (
    plot_class_distribution, plot_rr_distributions,
    plot_beat_windows, plot_correlation_heatmap, plot_pca_scatter
)

plt.style.use('dark_background')
cfg = load_config()

ModuleNotFoundError: No module named 'wfdb'

## 1. Dataset Overview

In [ ]:
records = get_records(cfg)
print(f'Records in {cfg["mode"]} mode: {records}')
meta_df = load_dataset_metadata(records)
print(f'\nDataset summary:')
print(meta_df.to_string(index=False))

## 2. Class Distribution

In [ ]:
segments = pd.read_parquet('../data/processed/segments.parquet')
print(f'Total beats: {len(segments):,}')
print(f'\nClass distribution:')
counts = segments['aami_label'].value_counts().reindex(AAMI_CLASSES, fill_value=0)
for cls, cnt in counts.items():
    print(f'  {cls}: {cnt:,} ({cnt/len(segments)*100:.1f}%)')

fig = plot_class_distribution(segments['label_int'], title='AAMI Class Distribution')
plt.savefig('../results/figures/eda_class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Beat Windows per Class

In [ ]:
beats_by_class = {}
for cls in AAMI_CLASSES:
    sub = segments[segments['aami_label'] == cls]
    beats_by_class[cls] = [np.array(sub.iloc[i]['filtered_window']) for i in range(min(5, len(sub)))]

fig = plot_beat_windows(beats_by_class)
plt.savefig('../results/figures/eda_beat_gallery.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. RR Interval Distribution

In [ ]:
fig = plot_rr_distributions(segments)
plt.savefig('../results/figures/eda_rr_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Feature Correlation and PCA

In [ ]:
X = pd.read_parquet('../data/processed/features.parquet')
meta = pd.read_parquet('../data/processed/meta.parquet')
y = meta['label_int'].values

print(f'Feature matrix: {X.shape}')
print(f'NaN values: {X.isna().sum().sum()}')

fig = plot_correlation_heatmap(X, top_n=25)
plt.savefig('../results/figures/eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

fig = plot_pca_scatter(X, y)
plt.savefig('../results/figures/eda_pca.png', dpi=120, bbox_inches='tight')
plt.show()